# Suite PAT — Patterns: symbols, variables, constraints, locators, expressions and collectors

A pattern (`Schemas.OfPattern`) is the shape of a frame (docs/STREAMING_QUERIES.md): its symbols, each standing for a
reference object and perhaps located by a locator, a value of the `Locators` schemas (docs/OBJECT_IDENTITY.md,
Locators), its variables, each standing for a value, and the constraints a frame meets, expressions of boolean result.
An expression (`Schemas.OfExpression`) ranges over a pattern; its terms are mbse-expressions'. A collector
(`Schemas.OfCollector`) folds a pattern's frames. Each is a schema object, built, validated and written as every one is;
frames, and matching, are the executor's (step 4e of the store overhaul).

In [ ]:
import { Builders as B, JSON as Json, Plain, Schemas as S } from "@mbse/schemas/Framework";
import { AttributeError, ValueError } from "@mbse/schemas/Framework/Errors";
import { repr } from "@mbse/schemas/Framework/Repr";
import { assert, equal, map, raises, same } from "./support.js";

/** wts.Features, a singleton directory owning definitions through `dir`, keyed by name, each owning its clauses
 * through `clauses`; and wts.Tags, a relation of three links, from a definition: built in `store`. */
function wts(store: any): any[] {
  const Text = store.Schemas.OfStr.as_schema();
  const Directory = store.Schemas.OfObject((o: any) => o.name("wts.Directory").ref(true).singleton("wts.Features"));
  const Definition = store.Schemas.OfObject((o: any) => o.name("wts.Definition").ref(true).properties([
    (t: any) => t.name("title").schema([Text]),
  ]));
  const Clause = store.Schemas.OfObject((o: any) => o.name("wts.Clause").ref(true));
  const Defines = store.Schemas.OfRelation((r: any) => (
    r
      .name("wts.Defines")
      .links([(t: any) => t.name("directory"), (t: any) => t.name("defn")])
      .properties([(t: any) => t.name("name").schema([Text])])
      .uniques([["directory", "name"], ["defn"]])
  ));
  const HasClause = store.Schemas.OfRelation((r: any) => (
    r
      .name("wts.HasClause")
      .links([(t: any) => t.name("defn"), (t: any) => t.name("clause")])
      .properties([(t: any) => t.name("name").schema([Text])])
      .uniques([["defn", "name"], ["clause"]])
  ));
  const Tags = store.Schemas.OfRelation((r: any) => r.name("wts.Tags").links(["defn", "tag", "by"].map((n) => (t: any) => t.name(n))));
  store.Schemas.OfObject(Directory, (o: any) => o.adjacencies([(t: any) => t.name("dir").relation([Defines]).me("directory")]));
  store.Schemas.OfObject(Definition, (o: any) => o.adjacencies([
    (t: any) => t.name("def").relation([Defines]).me("defn").owner(true),
    (t: any) => t.name("clauses").relation([HasClause]).me("defn"),
    (t: any) => t.name("tags").relation([Tags]).me("defn"),
  ]));
  store.Schemas.OfObject(Clause, (o: any) => o.adjacencies([(t: any) => t.relation([HasClause]).me("clause").owner(true)]));
  return [Directory, Definition, Clause, Defines, HasClause, Tags];
}

const store: any = new B.Store();
const [Text, Int, Bool] = [store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema(), store.Schemas.OfBool.as_schema()];
const [Directory, Definition, Clause, Defines, HasClause, Tags] = wts(store);

## PAT-01 · A pattern: symbols, variables and constraints, in order, each its own; validated

In [ ]:
const Edit = store.Schemas.OfPattern((p: any) => (
  p
    .name("wts.Edit")
    .description("A definition and one of its clauses")
    .symbols([
      (t: any) => t.name("defn").schema([Definition]),
      (t: any) => t.name("clause").schema([Clause]).description("C-1"),
    ])
    .variables([(t: any) => t.name("title").schema([Text]).description("its new title")])
    .requires([(c: any) => c.description("the title is new")])
));
assert(Edit.validate().length === 0 && same([...Edit.symbols.keys()], ["defn", "clause"]) && same([...Edit.variables.keys()], ["title"]));
assert(Edit.variables.get("title").schema === Text && Edit.symbols.get("clause").description === "C-1");
assert(Edit.requires.length === 1 && Edit.requires[0].result === Bool); // a constraint: an expression of the store's OfBool
assert(store.registered("wts.Edit") === Edit && store.module("wts").schemas.get("Edit") === Edit); // named: in its module
{
  const twin = new B.OfPattern.Builder(Edit, true, store).name("wts.Twin").clone();
  assert(twin.symbols.get("defn") !== Edit.symbols.get("defn") && twin.requires[0] !== Edit.requires[0]); // a clone owns its own
  assert(twin.variables.get("title") !== Edit.variables.get("title") && twin.variables.get("title").schema === Text);
}
const Port = store.Schemas.OfObject((o: any) => o.properties([(t: any) => t.name("pin").schema([Text])]));
{
  const bad = store.Schemas.OfPattern((p: any) => (
    p
      .symbols([(t: any) => t.name("x").schema([Port]), (t: any) => t.name("v")])
      .variables([(t: any) => t.name("v").schema([Definition]), (t: any) => t.name("w")])
      .parameters([(t: any) => t.name("w")])
  ));
  bad.variables.get("w").description = 1n; // what the builders refuse
  assert(same(bad.validate(), [
    "names used as both symbol and variable: ['v']",
    "names used as both variable and parameter: ['w']",
    "symbol 'x': a symbol stands for a reference object, so its schema is a reference object schema",
    "variable 'v': a reference object schema cannot be a property's schema",
    "variable 'w': a description is text, got int"]));
  assert(same(new S.OfPattern.Data({ symbols: new Map([["p", new S.OfSymbol.Data({ name: "p" })]]),
    parameters: new Map([["p", new S.OfParameter.Data({ name: "p" })]]) }).validate(), ["names used as both symbol and parameter: ['p']"]));
}

## PAT-02 · Locators: the shorthand, a path language, writes Locators data from a singleton or a symbol, hop by hop

In [ ]:
const Located = store.Schemas.OfPattern((p: any) => p.name("wts.Located").symbols([
  (t: any) => t.name("features").schema([Directory]).locator((l: any) => l.singleton(Directory)),
  (t: any) => t.name("defn").schema([Definition]).locator((l: any) => l.symbol("features", Directory).dir({ name: "DEF-693" })),
  (t: any) => t.name("clause").schema([Clause]).locator(
    (l: any) => l.singleton(store.wts.Directory).dir({ name: "DEF-693" }).clauses({ name: "C-1" })),
  (t: any) => t.name("any").schema([Clause]).locator((l: any) => l.symbol("defn", Definition).clauses()), // no keys: each
]));
assert(Located.validate().length === 0);
const root = Located.symbols.get("features").locator;
assert(root instanceof S.Locators.OfSingleton.Data && root.schema === Directory && root.next === null);
{
  const fromSymbol = Located.symbols.get("defn").locator;
  assert(fromSymbol instanceof S.Locators.OfSymbol.Data && fromSymbol.name === "features" && fromSymbol.schema === Directory);
  const hop = fromSymbol.next as any;
  assert(hop instanceof S.Locators.OfRelation.Data && hop.next === null);
  assert(hop.schema === Defines && hop.adjacency === "dir" && hop.link === "defn" && equal(hop.properties, map({ name: "DEF-693" })));
  const second = Located.symbols.get("clause").locator.next.next; // the object after a hop has one schema: its next hop is found
  assert(second.schema === HasClause && second.adjacency === "clauses" && second.link === "clause"
    && equal(second.properties, map({ name: "C-1" })));
  assert(Located.symbols.get("any").locator.next.properties.size === 0);
  // In full: a relation of more than two links, or after an object with no single schema.
  const tagged = store.Schemas.OfPattern((p: any) => p.symbols([
    (t: any) => t.name("defn").schema([Definition]),
    (t: any) => t.name("tag").locator((l: any) => l.symbol("defn", Definition).hop("tags", Tags, "tag", {})),
  ]));
  assert(tagged.validate().length === 0 && tagged.symbols.get("tag").locator.next.link === "tag");
  for (const [path, says] of [
    [(l: any) => l.symbol("defn", Definition).tags(), "'wts.Tags' has more than two links: give the hop in full"],
    [(l: any) => l.symbol("defn", Definition).hop("tags", Tags, "tag").dir(), "the object before 'dir' has no single schema"],
    [(l: any) => l.singleton(Directory).nope(), "'wts.Directory' has no adjacency 'nope': a locator traverses named adjacencies"],
    [(l: any) => l.hop("dir", Defines, "defn"), "a locator starts at a symbol or a singleton"],
    [(l: any) => l.nothing, "the object before 'nothing' has no single schema"],
  ] as [(l: any) => unknown, string][]) {
    raises([TypeError, AttributeError, ValueError], () => store.Schemas.OfSymbol((s: any) => s.locator(path)), says);
  }
  raises(ValueError, () => B._LocatorPath(store)._built(), "a locator starts at a symbol or a singleton");
  assert(repr(B._LocatorPath(store)) === "<locator>");
  assert(B._LocatorPath(store).singleton(Directory).hop("dir", Defines, "nope")._current === null); // no such link: nothing after
  assert(B._LocatorPath(store).singleton(Directory).hop("dir", Directory, "x")._current === null); // nor through no relation
  // The member is the symbol's `locator` adjacency too, as every adjacency is: linked, visited, or written in full.
  const built = store.Schemas.OfSymbol((s: any) => s.locator([root]));
  assert(built.locator === root);
  const seen: string[][] = [];
  store.Schemas.OfSymbol(built, (s: any) => s.locator((a: any) => {
    seen.push([...a].map((e: any) => e.locator.as_schema().name));
    return a.as_iterable((i: any) => i);
  }));
  assert(equal(seen, [["Locators.OfSingleton"]]));
  store.Schemas.OfSymbol(built, (s: any) => s.locator((a: any) => a.remove((e: any) => e)));
  assert(built.locator === null);
  const direct = store.Locators.OfRelation((h: any) => h.schema([Defines]).adjacency("dir").link("defn").properties({ name: "D" }));
  assert(store.Locators.OfSymbol((r: any) => r.name("x").schema([Directory]).next([direct])).next === direct);
  assert(new B.Locators.OfRelation.Builder(direct, true, store).properties(null).clone().properties.size === 0); // cleared
  assert(B.Locators.OfSingleton.Builder === B.LocatorSingletonBuilder && S.Locators.OfSingleton.Builder === B.LocatorSingletonBuilder);
}

## PAT-03 · What validation refuses of a locator: its root, its symbol, its singleton, each hop's relation, adjacency, link and properties

In [ ]:
{
  /** A pattern's problems with a symbol `s` located by `locator`, after the symbols `before`, each a Definition. */
  const problems = (locator: unknown, before = ["defn"]) => {
    const symbols = new Map(before.map((name) => [name, new S.OfSymbol.Data({ name, schema: Definition })]));
    symbols.set("s", new S.OfSymbol.Data({ name: "s", locator: locator as any }));
    return new S.OfPattern.Data({ symbols }).validate();
  };
  const hop = (schema: unknown, adjacency: string, link: string, properties: Record<string, unknown> = {}, next: unknown = null) =>
    new S.Locators.OfRelation.Data({ schema, adjacency, link, properties: new Map(Object.entries(properties)), next });
  assert(same(problems(hop(Defines, "dir", "defn")), [
    "symbol 's': a locator starts at a symbol or a singleton: Locators.OfSymbol or Locators.OfSingleton"]));
  assert(same(problems(new S.Locators.OfSymbol.Data({ name: "later", schema: Definition })), [
    "symbol 's': a locator starts at symbol 'later', which the pattern does not declare before it"]));
  assert(same(problems(new S.Locators.OfSymbol.Data({ name: "defn", schema: Port })), [
    "symbol 's': a locator's symbol has a reference object schema"]));
  assert(same(problems(new S.Locators.OfSingleton.Data({ schema: Definition })), [
    "symbol 's': a locator's singleton is a singleton's schema"]));
  const at = (next: unknown) => new S.Locators.OfSingleton.Data({ schema: Directory, next });
  assert(same(problems(at(new S.Locators.OfSingleton.Data({ schema: Directory }))), [
    "symbol 's': hop 1: only a locator's root is a symbol or a singleton; each hop is a Locators.OfRelation"]));
  assert(same(problems(at(hop(Directory, "dir", "defn"))), ["symbol 's': hop 1: a hop's schema is a relation"]));
  assert(same(problems(at(hop(HasClause, "dir", "defn"))), [
    "symbol 's': hop 1: 'wts.Directory' has no adjacency 'dir' to 'wts.HasClause': a locator traverses named adjacencies"]));
  assert(same(problems(at(hop(Defines, "nope", "defn"))), [
    "symbol 's': hop 1: 'wts.Directory' has no adjacency 'nope' to 'wts.Defines': a locator traverses named adjacencies"]));
  assert(same(problems(at(hop(Defines, "dir", "directory"))), [
    "symbol 's': hop 1: 'directory' is not a link of 'wts.Defines' to the next object"]));
  assert(same(problems(at(hop(Defines, "dir", "defn", { name: "D", colour: "red" }))), [
    "symbol 's': hop 1: 'wts.Defines' has no properties ['colour']"]));
  // After a hop whose link has no schema of its own, the adjacency is not checked; the hop after it still is.
  assert(same(problems(at(hop(Defines, "dir", "defn", {}, hop(Tags, "anything", "nope")))), [
    "symbol 's': hop 2: 'nope' is not a link of 'wts.Tags' to the next object"]));
  const link = new S.OfRelation.Data({ entry: new S.OfEntry.Data({ links: new Map([["a", new S.OfSymbol.Data({ name: "a" })],
    ["b", new S.OfSymbol.Data({ name: "b", locator: at(null) })]]) }) });
  assert(same(link.validate(), ["link 'b': a relation's link has no locator; a pattern's symbol may"]));
}

## PAT-04 · Expressions and constraints: a pattern and a result schema; a constraint's result is OfBool; a pattern requires its own constraints

In [ ]:
const Count = store.Schemas.OfExpression((e: any) => e.name("wts.Count").pattern([Edit]).result([Int]));
assert(Count.validate().length === 0 && Count.pattern === Edit && Count.result === Int);
assert(store.registered("wts.Count") === Count && Count.equals(Count) && !Count.equals(Edit));
{
  const Holds = store.Schemas.OfConstraint((c: any) => c.name("wts.Holds").pattern([Edit]));
  assert(Holds.result === Bool && Holds.validate().length === 0 && Holds instanceof S.OfExpression.Data);
  assert(new B.OfConstraint.Builder().create().result.name === "Schemas.OfBool"); // outside a store, the template's
  assert(same(new S.OfExpression.Data({ pattern: Count }).validate(), ["an expression's pattern is a pattern",
    "an expression needs a result schema"]));
  assert(same(new S.OfExpression.Data({ result: Port, name: "x y" }).validate(), [
    "name 'x y' is not identifiers separated by dots, e.g. 'crm.Contact'"]));
  assert(same(new S.OfExpression.Data({ result: Definition }).validate(), [
    "result: a reference object schema cannot be a property's schema"]));
  const own = new S.OfExpression.Data({ result: Bool });
  const mine = new S.OfPattern.Data({ requires: [own] });
  own.pattern = mine;
  assert(mine.validate().length === 0); // over its own pattern
  assert(same(new S.OfPattern.Data({ requires: [Count] }).validate(), [
    "requires[0] is a constraint: an expression whose result is Schemas.OfBool"]));
  assert(same(new S.OfPattern.Data({ requires: [Holds] }).validate(), ["requires[0] is over this pattern's frames, not another's"]));
  assert(same(new S.OfPattern.Data({ requires: [new S.OfExpression.Data({ result: Bool, description: 2n as never })] }).validate(), [
    "requires[0]: a description is text, got int"]));
}

## PAT-05 · Collectors: a pattern, the names it collects, commutative or associative, two suffixes, and what it computes

In [ ]:
{
  const Sum = store.Schemas.OfCollector((c: any) => (
    c
      .name("wts.Sum")
      .pattern([Edit])
      .symbols(["clause"])
      .variables(["title"])
      .commutative(true)
      .associative(true)
      .suffixes({ current: "_a", next: "_b" })
      .computes([Count])
  ));
  assert(Sum.validate().length === 0 && same(Sum.symbols, ["clause"]) && same(Sum.variables, ["title"])
    && equal(Sum.suffixes, map({ current: "_a", next: "_b" })));
  assert(same(new B.OfCollector.Builder(Sum, false, store).symbols._held(), ["clause"]) && Sum.computes === Count && Sum.equals(Sum));
  const cleared = new B.OfCollector.Builder(Sum, true, store).name("wts.Cleared").symbols(null).variables(null).suffixes(null).clone();
  assert(cleared.symbols.length === 0 && cleared.variables.length === 0 && cleared.suffixes === null);
  const C = (fields: Record<string, unknown>) => new S.OfCollector.Data(fields as never).validate();
  assert(same(C({}), [
    "a collector needs a pattern",
    "a collector is commutative or associative, so that its result does not depend on the order frames arrive in"]));
  assert(same(C({ pattern: Edit, symbols: ["nope"], variables: ["none"], associative: true }), [
    "it collects symbols ['nope'], which its pattern does not declare",
    "it collects variables ['none'], which its pattern does not declare"]));
  assert(same(C({ pattern: Edit, commutative: true, suffixes: map({ current: "x" }) }), [
    "a collector's suffixes are 'current' and 'next'"]));
  assert(same(C({ pattern: Edit, commutative: true, suffixes: map({ current: "x", next: "x" }) }), [
    "a collector's 'current' and 'next' suffixes differ"]));
  assert(same(C({ pattern: Edit, commutative: true, computes: Edit }), ["what a collector computes is an expression"]));
  assert(same(C({ pattern: Edit, commutative: true, computes: new S.OfExpression.Data() }), [
    "computes: an expression needs a result schema"]));
  assert(new B.OfCollector.Builder(store.Schemas.OfCollector((c: any) => c), false, store).suffixes._held() === null);
}

## PAT-06 · Patterns, expressions, collectors and locators in a store's snapshot: written once, read back the same

In [ ]:
{
  const log = Plain.ToPlain(store).Snapshot() as any[];
  const back: any = new B.Store();
  Plain.FromPlain(back).Snapshot(log);
  assert(Json.dumps(Plain.ToPlain(back).Snapshot() as never) === Json.dumps(log as never));
  assert(Json.dumps(Json.loads(Json.ToJSON(back).Snapshot())) === Json.dumps(log as never));
  assert(equal(back.registered("wts.Located").symbols.get("clause").locator.next.next.properties, map({ name: "C-1" })));
  const read = back.registered("wts.Edit");
  assert(read !== Edit && same([...read.symbols.keys()], ["defn", "clause"]) && read.requires[0].result === back.Schemas.OfBool.as_schema());
  assert(back.registered("wts.Sum").pattern === read && back.registered("wts.Sum").computes === back.registered("wts.Count"));
  const hops = log.filter((e: any) => e.has("exists")
    && String(e.get("exists").get("definition").get("key") ?? "").startsWith("Locators.OfRelation/"));
  assert(hops.length === 4 && hops.every((e: any) => e.get("exists").get("definition").get("properties").get("adjacency"))); // each hop, written once
}